# pilot_paper 结果闭合入口

该 Notebook 用于在 Colab 中完成 pilot_paper 结果闭合: 挂载 Google Drive、拉取当前仓库、从 `/content/drive/MyDrive/SLM/pilot_paper_results` 物化前序结果包、重建 attack matrix、external baseline formal import、internal ablation、pilot_paper fixed-FPR 共同协议记录, 并把完整结果包写回 Google Drive。

Notebook 只调度 `scripts/` 中的 repository commands, 不直接手写正式 records、tables、figures 或 reports。


## 运行前准备

1. 确认前序方法主流程和 baseline Notebook 已把结果包写入 `/content/drive/MyDrive/SLM/pilot_paper_results/`。
2. 该入口不需要 GPU, 但需要能够访问 Google Drive。
3. 如果本次需要干净复盘, 应先清理 `/content/drive/MyDrive/SLM/pilot_paper_results/` 后重新运行前序 Notebook, 再运行本入口。
4. 输出完整结果包默认写入 `/content/drive/MyDrive/SLM/pilot_paper_results/complete_result_package`。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
os.environ.setdefault('SLM_WM_PROTOCOL_PROFILE', 'pilot_paper_fixed_fpr_0_01')
os.environ.setdefault('SLM_WM_PROMPT_SET', 'pilot_paper')
os.environ.setdefault('SLM_WM_PROMPT_FILE', 'configs/paper_main_pilot_paper_prompts.txt')
os.environ.setdefault('SLM_WM_PILOT_PAPER_PACKAGE_SEARCH_ROOT', '/content/drive/MyDrive/SLM/pilot_paper_results')
os.environ.setdefault('SLM_WM_PILOT_PAPER_COMPLETE_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/SLM/pilot_paper_results/complete_result_package')
os.environ.setdefault('SLM_WM_PILOT_PAPER_TARGET_FPR', '0.01')
print({
    'protocol_profile': os.environ['SLM_WM_PROTOCOL_PROFILE'],
    'prompt_set': os.environ['SLM_WM_PROMPT_SET'],
    'prompt_file': os.environ['SLM_WM_PROMPT_FILE'],
    'package_search_root': os.environ['SLM_WM_PILOT_PAPER_PACKAGE_SEARCH_ROOT'],
    'complete_drive_output_dir': os.environ['SLM_WM_PILOT_PAPER_COMPLETE_DRIVE_OUTPUT_DIR'],
    'target_fpr': os.environ['SLM_WM_PILOT_PAPER_TARGET_FPR'],
})


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')
if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)
subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
from pathlib import Path

package_search_root = Path(os.environ['SLM_WM_PILOT_PAPER_PACKAGE_SEARCH_ROOT'])
assert package_search_root.exists(), f'未找到 pilot_paper Google Drive 结果目录: {package_search_root}'
zip_count_by_dir = {
    child.name: len(list(child.glob('*.zip')))
    for child in sorted(package_search_root.iterdir())
    if child.is_dir()
}
print({'package_search_root': str(package_search_root), 'zip_count_by_dir': zip_count_by_dir})


In [ ]:
import os
import subprocess
import sys

package_search_root = os.environ['SLM_WM_PILOT_PAPER_PACKAGE_SEARCH_ROOT']
complete_drive_output_dir = os.environ['SLM_WM_PILOT_PAPER_COMPLETE_DRIVE_OUTPUT_DIR']
target_fpr = os.environ['SLM_WM_PILOT_PAPER_TARGET_FPR']

commands = [
    [sys.executable, 'scripts/write_pilot_paper_result_records.py', '--package-search-root', package_search_root, '--materialize-only'],
    [sys.executable, 'scripts/write_attack_matrix_outputs.py'],
    [sys.executable, 'scripts/write_primary_baseline_result_candidates.py', '--target-fpr-override', target_fpr],
    [sys.executable, 'scripts/write_primary_baseline_formal_import_protocol.py'],
    [sys.executable, 'scripts/write_external_baseline_comparison_outputs.py'],
    [sys.executable, 'scripts/write_internal_ablation_outputs.py'],
    [sys.executable, 'scripts/write_pilot_paper_result_records.py', '--package-search-root', package_search_root, '--require-existing-evidence'],
    [
        sys.executable,
        'scripts/write_pilot_paper_fixed_fpr_common_protocol_outputs.py',
        '--candidate-records-path',
        'outputs/pilot_paper_fixed_fpr_results/pilot_paper_result_records.jsonl',
        '--require-existing-evidence',
    ],
    [
        sys.executable,
        'scripts/write_pilot_paper_complete_result_package.py',
        '--package-search-root',
        package_search_root,
        '--drive-output-dir',
        complete_drive_output_dir,
    ],
]

for command in commands:
    print('run_repository_command', ' '.join(command))
    subprocess.run(command, check=True)


In [ ]:
from pathlib import Path

summary_paths = [
    Path('outputs/pilot_paper_fixed_fpr_results/pilot_paper_result_record_summary.json'),
    Path('outputs/pilot_paper_fixed_fpr_common_protocol/pilot_paper_common_protocol_summary.json'),
    Path('outputs/pilot_paper_complete_result_package/pilot_paper_complete_package_summary.json'),
]
for summary_path in summary_paths:
    print(summary_path)
    print(summary_path.read_text(encoding='utf-8')[:4000])

complete_archives = sorted(Path(os.environ['SLM_WM_PILOT_PAPER_COMPLETE_DRIVE_OUTPUT_DIR']).glob('pilot_paper_complete_result_package_*.zip'))
assert complete_archives, '未找到写回 Google Drive 的 pilot_paper 完整结果包'
print({'latest_complete_archive': str(complete_archives[-1])})
